In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm
import cv2

import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms.v2 as v2
from torchvision import models
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import autocast, GradScaler
from torchmetrics import Accuracy

In [73]:
# # https://drive.google.com/file/d/16QCd2dhOGNNDFxTFzLFmBG9SzP06A-y5/view
# !gdown 16QCd2dhOGNNDFxTFzLFmBG9SzP06A-y5
# !unzip ./data.zip

In [74]:
seed = 43
torch.random.manual_seed(seed)

device = "cuda" if torch.cuda.is_available() else "cpu"

batch_size = 16
num_epochs = 10
lr = 1e-5

root_path = "/home/stefan/ioai-prep/kits/roai-2025/angry-birds/data"

In [75]:
weights = models.ResNet50_Weights.IMAGENET1K_V1
resnet = models.resnet50(weights=weights).to(device)
resnet.fc = nn.Identity()
resnet.eval()
print("model loaded.")

model loaded.


# Data preparation

# Model

In [82]:
class BirdsNet(nn.Module):
  def __init__(self):
    super().__init__()

    self.head = nn.Sequential(
      nn.Dropout(0.4),
      nn.BatchNorm1d(1),
      nn.Linear(2048, 1),
    )

  def forward(self, x):
    # features = self.backbone(x)
    logits = self.head(x)
    return logits

In [ ]:
model = BirdsNet().to(device)

# model(batch[0]).shape

torch.Size([16, 1, 1])

# Training

In [ ]:
# # warming up
# for loader in [train_loader, val_loader]:
#   for _ in tqdm(loader):
#     pass

100%|██████████| 75/75 [00:00<00:00, 238.08it/s]


# Submission

In [89]:
test_dataset = AngryBirdsDataset("test")
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

df_test = pd.read_csv(f"{root_path}/test.csv")

6993 from 6993


In [90]:
answer = []

model.eval()
for batch in tqdm(test_loader):
  batch = batch.to(device)
  with torch.no_grad():
    logits = model(batch).squeeze(-1)
  preds = (torch.sigmoid(logits) >= 0.5).detach().cpu().long().squeeze(-1)
  answer.extend(preds.tolist())

100%|██████████| 438/438 [00:01<00:00, 230.56it/s]


In [91]:
submission = pd.DataFrame({
    "datapointID": df_test["datapointID"],
    "subtaskID": 1,
    "answer": answer
})

submission.head()

,datapointID,subtaskID,answer
0,0,1,0
1,1,1,0
2,2,1,1
3,3,1,0
4,4,1,1


In [ ]:
submission.to_csv(f"{root_path}/submission.csv", index=False)